# Advanced Statistical Inference and Hypothesis Testing Toolkit
**Dataset:** Cleaned Superstore transaction data  
**Objective:** Demonstrate maximum likelihood estimation (MLE), confidence intervals, multi-group hypothesis testing, and interpretation.

> Run this notebook from top to bottom. Upload `superstore_cleaned_for_analytics.csv` when prompted, or place it in the same folder as this notebook.

## 1. Imports and data ingestion

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from pathlib import Path

sns.set_theme(style="whitegrid")

In [ ]:
DATA_FILE = Path("superstore_cleaned_for_analytics.csv")
if not DATA_FILE.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        DATA_FILE = Path(next(iter(uploaded.keys())))
    except Exception as exc:
        raise FileNotFoundError(
            "Place superstore_cleaned_for_analytics.csv beside this notebook "
            "or upload it when prompted."
        ) from exc

df = pd.read_csv(DATA_FILE, parse_dates=["Order Date", "Ship Date"], low_memory=False)
df["Sales"] = pd.to_numeric(df["Sales"], errors="coerce")
df["Profit"] = pd.to_numeric(df["Profit"], errors="coerce")
df = df.dropna(subset=["Sales", "Profit", "Region", "Category"])
df.head()

## 2. Descriptive statistics and distribution checks

In [ ]:
print("Rows:", len(df))
display(df[["Sales", "Profit", "Quantity", "Discount"]].describe().T)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Sales"], bins=40, kde=True, ax=axes[0])
axes[0].set_title("Sales distribution")
sns.histplot(df["Profit"], bins=40, kde=True, ax=axes[1])
axes[1].set_title("Profit distribution")
plt.tight_layout()

## 3. Maximum Likelihood Estimation (MLE)

For a normal model, the MLEs are the sample mean and the maximum-likelihood variance (denominator *n*, not *n−1*). We estimate parameters for order-level sales as a teaching example. Sales may be skewed, so interpret the normal fit cautiously.

In [ ]:
x = df["Sales"].dropna().to_numpy(dtype=float)
mu_mle = x.mean()
sigma2_mle = ((x - mu_mle) ** 2).mean()
sigma_mle = np.sqrt(sigma2_mle)
print(f"Normal MLE mean: {mu_mle:,.3f}")
print(f"Normal MLE standard deviation: {sigma_mle:,.3f}")

# Verify by maximizing the negative log-likelihood numerically.
def negative_log_likelihood(params, values):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    return -np.sum(stats.norm.logpdf(values, loc=mu, scale=sigma))

result = minimize(
    negative_log_likelihood,
    x0=[mu_mle, np.log(sigma_mle)],
    args=(x,),
    method="Nelder-Mead"
)
print("Numerical MLE:", result.x[0], np.exp(result.x[1]))

## 4. Confidence interval for mean sales

In [ ]:
n = len(x)
mean_sales = x.mean()
se = stats.sem(x)
ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=mean_sales, scale=se)
print(f"95% t-confidence interval for mean sales: [{ci_low:,.2f}, {ci_high:,.2f}]")

# Bootstrap confidence interval (resampling orders with replacement).
rng = np.random.default_rng(42)
B = 2000
boot_means = np.array([rng.choice(x, size=n, replace=True).mean() for _ in range(B)])
boot_ci = np.quantile(boot_means, [0.025, 0.975])
print(f"95% bootstrap CI: [{boot_ci[0]:,.2f}, {boot_ci[1]:,.2f}]")

## 5. Two-sample hypothesis test: Consumer vs Corporate segment

**Question:** Is average sales different between Consumer and Corporate orders?  
**H₀:** The population mean sales are equal.  
**H₁:** The population mean sales differ.  
Welch's t-test is used because it does not assume equal population variances.

In [ ]:
consumer = df.loc[df["Segment"] == "Consumer", "Sales"].dropna()
corporate = df.loc[df["Segment"] == "Corporate", "Sales"].dropna()
welch = stats.ttest_ind(consumer, corporate, equal_var=False, alternative="two-sided")
print("Consumer n / mean:", len(consumer), consumer.mean())
print("Corporate n / mean:", len(corporate), corporate.mean())
print("Welch t statistic:", welch.statistic)
print("p-value:", welch.pvalue)
alpha = 0.05
print("Decision:", "Reject H0" if welch.pvalue < alpha else "Fail to reject H0")

## 6. Multi-sample hypothesis test: Sales across regions

**H₀:** Mean sales are equal across all regions.  
**H₁:** At least one regional mean differs.  
We report one-way ANOVA and the non-parametric Kruskal–Wallis test as a robustness comparison. A significant omnibus test does not identify which pairs differ; post-hoc analysis would be a separate step.

In [ ]:
region_groups = [
    group["Sales"].dropna().to_numpy()
    for _, group in df.groupby("Region")
]
anova = stats.f_oneway(*region_groups)
kruskal = stats.kruskal(*region_groups)
print("One-way ANOVA F:", anova.statistic, "p:", anova.pvalue)
print("Kruskal-Wallis H:", kruskal.statistic, "p:", kruskal.pvalue)

plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Region", y="Sales")
plt.title("Order sales by region")
plt.tight_layout()

## 7. Interpretation and limitations

- A p-value below 0.05 is evidence against the corresponding null hypothesis; it is not the probability that the null hypothesis is true.
- Confidence intervals quantify uncertainty in an estimated population mean under the sampling assumptions.
- The dataset is observational and may contain repeated customers, skewed outcomes, and business-seasonality effects. Statistical significance does not establish causality.
- The normal MLE is a model-based summary; inspect distribution shape and consider transformations or alternative distributions before using it for operational decisions.
- If orders are not independent, use a design that accounts for customer/order clustering.

## 8. Conclusion template

After running the notebook, summarize: (1) the MLE estimates, (2) the 95% confidence interval, (3) the Welch test result and practical difference in group means, and (4) the regional ANOVA/Kruskal–Wallis result. Include the actual output values from your run rather than inventing conclusions.